In [ ]:
# Course notation: find Latex_macros.ipynb in this folder or in any parent folder, so the cell works wherever the notebook sits
import pathlib
_here = pathlib.Path.cwd().resolve()
_macros = next((d / "Latex_macros.ipynb" for d in [_here, *_here.parents] if (d / "Latex_macros.ipynb").exists()), None)
if _macros is not None:
    get_ipython().run_line_magic("run", f'"{_macros}"')
else:
    print("Latex_macros.ipynb not found: math may show without the course notation (harmless in this assignment)")

<html>
<p style="font-size:32px"><strong>SFT practice: Fine-tuning a pre-trained model</strong></p>
</html>

# Assignment

We have spent some time in class talking about Transfer Learning and Fine Tuning an LLM.

The objective of this practice assignment is for you to demonstrate a depth of knowledge of the topic.

In this assignment you will fine-tune a Pre-Trained model in a variety of ways.

You are to evaluate each method, provide commentary and demonstrate some insight into the process.

## Project design

Coding projects in the Coding Assistant era become a challenge as a means of differentiating the depth of
knowledge that a student has acquired.  The project design is an attempt to measure the depth of your knowledge.

I fully expect you to use an AI Assistant/Agent to complete the project.
My experience is that doing so will not present much of an intellectual challenge to you:
the result in high quality code

But everyone else's code will be high quality too. You will need to find a way to distinguish your work from
every else's.

The tasks that you are asked to perform are the
*minimum* that you are required to perform and will account for $\frac{2}{3}$ of the total points
that are possible to earn for this part of the Project.

The other $\frac{1}{3}$ of the total points that are possible will be at the grader's discretion.
It will be based on
- your performance of tasks of your own choice that are *in addition* to the *minimum*
    - the minimum submission is just that, a minimum
    - please consider: what else can I do to demonstrate deep knowledge
        - propose a new task, justify why it is interesting, perform it, evaluate/compare to the other tasks

Additionally: the instructions in this notebook are intentionally minimal (to provide less detail to your coding assistant). Please ask questions in person, in class and to the TA, if anything is not clear.

## Format of your submission

Your submission will be a *well structured* Jupyter notebook that is clearly presented.
- Follow the Recipe for Machine Learning that we discuss in the FRE 7773 course
- **Hand in the notebook already executed, with every output saved.** The grader will *not* run your notebook (that does not scale): the grader reads the saved outputs and your `submission/` files
    - before you submit, choose "Restart kernel and run all cells" yourself, and check that it runs to the end without error
    - a notebook with missing or stale outputs cannot be graded
- The TA should be able to *easily* find
    - your results for each required task
    - the answer to any question that is posed as part of the task
    - your unique "above and beyond" contribution
        - clearly explain what you are doing
        - clearly explain why this is interesting
        - perform your experiment
        - analyze the results and report any insights/things that you learned
        - even a "failed" experiment can be interesting if you learned or discovered something

Since there are multiple parts: please use Section headings to assist the TA in identifying each part.

## Requirements

You may use **any toolkit** you like. PyTorch is the easiest choice; Keras is allowed but not recommended. Your coding assistant may write the code.

The point of Models 2 to 6 is to learn how fine-tuning works, so in those models the training loop and the changes to the model are written *by you or your assistant*
- **no tool that runs the training for you** (for example the HuggingFace `Trainer`)
- **no tool that builds or inserts the new head or the adapters for you** (for example `peft`)
- you may load the pre-trained model with any tool, and use ordinary libraries (optimizers, data loaders, metrics)

**Model 7 is the exception:** after you have shown that you can do it by hand, use the full power of the toolkits (`Trainer`, `peft`, ...).

A GPU makes things faster, but the model is small enough to work without one. The free tier of Google Colab is enough.

You will use the [Financial Phrasebank](https://www.researchgate.net/publication/251231364_FinancialPhraseBank-v10) dataset (available on HuggingFace)
- the flavor to use is "sentences_allagree"

## Given code and the fixed split

The file `sft_given.py` is given: it loads Financial PhraseBank `sentences_allagree` and cuts it into a **fixed** train / validation / test split
(the same for everybody). Use it, and do not edit it. Labels: 0 = negative, 1 = neutral, 2 = positive.

In [ ]:
# Given: load the helper file and the fixed split (do not edit sft_given.py, and do not make your own split)
%load_ext autoreload
%aimport sft_given
import sft_given as G
train, val, test = G.load_split()
print(len(train), len(val), len(test), "examples in train / validation / test")

# The task

You will fine tune a pre-trained model 
- `distilbert-base-uncased`

(which has been trained on general language) to understand "financial language".

Specifically, you will fine tune the pre-trained model to solve
- Sentiment classification on financial language
- using the Financial PhraseBank dataset (available on HuggingFace)

The minimum requirement is to
- create a number of models
- evaluate each
- compare

using different types of fine-tuning.

Remember: you are encouraged to do more than the minimum !

# Before the models: look at the data

**Your notebook must be in the style of the [Recipe for Machine Learning](https://github.com/kenperry-public/ML_Fall_2022/blob/master/Recipe_Overview.ipynb).**
- treat it as a movie of your work: the steps in order, with Markdown explaining what you do and why
- include steps that did not work, and what you learned from them
- the grader reads your saved outputs without running anything, so each result must be visible in the notebook and easy to find

The Recipe begins with the data, so begin there.
- Explore the training data: what does it look like, and what do you notice?
- Decide how you will judge a model. **Is one number enough?** Justify your choice of metrics.
- Report the same metrics for every model, and finish with a single table in the notebook comparing all seven
    - include the number of trainable weights and the training time

Keep it short: a few cells and a few sentences.

## Words used in this assignment

- **"Starting with Model X"** means: make a *copy* of Model X's trained weights (a deep copy) and continue from the copy. Keep Model X itself unchanged, because you will compare with it later.
- **Trainable weights** means the parameters that are not frozen (`requires_grad` is True): biases and LayerNorm weights count if they are trained, and a frozen head does not count. Count them from the model, in code.
- **The head** of the HuggingFace DistilBERT classifier is the two layers `pre_classifier` and `classifier`, plus what the model's `forward` does with them.
- **Model 1 trains nothing** (its untrained head is not updated by anything), so report 0 trainable weights and 0 seconds for it.
- **Seconds** is the training time of that model's own stage only (Model 4 does not include Model 2's time).
- **Models you compare outputs of** (for example the identity check in Model 6) must be in evaluation mode, so that dropout does not change them; a difference below 1e-5 counts as the same.
- Your LoRA scaling (alpha) is your choice; say what you used.
- Choose the number of epochs, the learning rate and the batch size yourself, and report them. Use `val` to choose and `test` only to report.

# Minimum tasks

## Model 1: The Pre-trained model

Instantiate the pre-trained model together with an appropriate classifier head.
Evaluate it before any training: this is your baseline. (The head is random, so near-chance results are expected.)

## Model 2: Fine-tune only the Classifier head

Starting with Model 1
- freeze all the weights *except* for the weights in the Classifier head
    - the head may be more than one layer
- fine-tune just the weights of the Classifier head

## Model 3: Your own Classifier head

The purpose of this part is for you to demonstrate competency in the Functional architecture.

Starting with the pre-trained model (Model 1)
- *remove* the existing Classifier head: its computation, not only its layers (check what the model does after the body)
- *replace* it with a Classifier head of your own design
- freeze all the weights *except* for the newly added Classifier head
- fine-tune just the weights in the newly added Classifier head

It is *required* for you
- to create a Classifier head that is different than the original
- *explain* why it is different and what inspired your design choices
- *explain* the technical challenge of grafting the head onto the pre-existing body
- you may **not** use a tool that builds the new head for you: write it yourself from ordinary layers
    - you may load the pre-trained model (Model 1) with any tool, and your new model may reuse a copy of its body

Compare with Model 2 in a sentence or two.

## Model 4: Then fine-tune the body

Starting with Model 2 (the trained head)
- freeze the weights of the Classifier head
- fine-tune all other weights

## Model 5: Fine-tune everything at once

Starting again with the pre-trained model (Model 1), with its *untrained* head
- fine-tune all the weights, head and body, simultaneously

Compare Models 2, 4 and 5. Why might the order in which you train the parts matter?

## Model 6: LoRA on the FFN blocks, by hand

Starting with Model 2 (the trained head)
- freeze the weights of the Classifier head
- fine-tune the other weights *using LoRA* with the following restrictions
    - you will fine-tune **only the weights** of the FFN blocks in every layer
        - this is where we hypothesize "world knowledge" acquired during training is stored
        - each FFN block has 2 `Dense` layers: `lin1`, `lin2`
            - apply LoRA **to each** of the 2 `Dense` layers
        - use rank 8 for the LoRA adapters

Additional requirements
- the objective is for you to demonstrate competence with the Functional architecture
    - you must *manually* create the adapters and insert them into the model
        - you **may not** use any tool that automatically does this (e.g. HuggingFace has PEFT tools)
- you should manually calculate the number of trainable weights in this configuration (the adapter weights you train; the frozen head does not count)
    - report this number, along with your calculation
    - verify that your model matches this calculation
- verify that the model with freshly created adapters gives the *same outputs* as the model without them (before any training)
    - explain why your adapter initialization makes this true

## Model 7: The easy way, with the toolkits

You have now done it by hand. Here there are **no restrictions on tools**: let a toolkit do the work.

Starting with Model 2 (the trained head)
- fine-tune with LoRA on `lin1` and `lin2` of every FFN block, rank 8, head frozen: **the same configuration as Model 6**
- for example, `peft` to insert the adapters and the HuggingFace `Trainer` to train
- use the same LoRA scaling (alpha) as in Model 6, so that the comparison is fair
- check what the toolkit made trainable: with the head frozen, the count must equal Model 6's

Compare with Model 6, in a few sentences
- do you get the same number of trainable weights? About the same accuracy? How long did each take?
- what did the toolkit do for you that you wrote by hand?

## Going beyond the minimum

This part is yours to design: propose a question, say why it is interesting, test it fairly, and report what you learned.
A few hints, nothing more:
- the dataset comes in other flavors
- look closely at the data: is there a pattern that a model could use?
- the adapters could go in other places, or be bigger or smaller
- a model can be right or wrong for different reasons
- the same recipe might work on a different kind of model

# Technical advice


HuggingFace has dropped most support for Keras models and tools.

PyTorch is the path of least resistance: use it unless you have a reason not to.

## Grading

- The **minimum tasks** (the data exploration, Models 1 to 7, the comparison table and the Submission Summary) account for **two thirds** of the grade for this assignment.
- **Going above and beyond** accounts for the other **one third**, at the grader's discretion.
- This is a practice run for the larger RFT project: keep it simple, and ask if anything is unclear.

# How to submit, and the Submission Summary (required)

The grader reads your results from fixed files, whatever toolkit you used. Your notebook is the narrative; the files are the facts.

1. Use the given `sft_given.py` (same folder as this notebook, do not edit it) to get the **fixed split**: `train, val, test = load_split()`.
    - everyone uses the same seed and the same test set, so results are comparable and the grader can check them
    - tune on `val`; touch `test` only to report results: using the test labels for anything else is an honesty violation
2. For each model, predict the label of every example in `test`, in order.
3. Run the last cell: `write_submission(...)` writes `results.csv`, `predictions.csv` and `environment.txt` into `submission/`;
   `check_submission(...)` re-reads them and prints the summary.
4. Hand in: this notebook **executed, with all outputs saved** (restart the kernel and run all cells yourself first; the grader will not run it), plus the `submission/` folder, in one zip file.
    - your explanations go in the notebook, under each model: a few lines are enough

A missing summary, or one that prints `SOME CHECKS FAILED`, costs 10 points.

In [ ]:
# Run this cell LAST. Fill in the values from YOUR notebook (counted from your models, not typed from memory).
import sft_given as G
train, val, test = G.load_split()                  # the fixed split: do not make your own

predictions = {                                    # one predicted label (0, 1, 2) per test example, in test order
    "model_1": None,                               # Model 1: pre-trained model, untrained head
    "model_2": None,                               # Model 2: classifier head fine-tuned
    "model_3": None,                               # Model 3: your own classifier head
    "model_4": None,                               # Model 4: head first, then the body
    "model_5": None,                               # Model 5: everything at once
    "model_6": None,                               # Model 6: your manual LoRA
    "model_7": None,                               # Model 7: LoRA with the toolkits (peft, Trainer, ...)
}
trainable_params = {f"model_{k}": None for k in range(1, 8)}     # counted from the models (0 for Model 1)
expected_trainable_lora = None                     # your hand calculation for Model 6 (Model 7 has the same configuration)
adapters_identity_at_init = None                   # True if the fresh adapters changed no output before training
seconds = {f"model_{k}": None for k in range(1, 8)}               # training time of each model, in seconds
toolkit = ""                                       # e.g. "PyTorch + transformers: own loops for Models 1-6; peft + Trainer for Model 7"

G.write_submission("submission", predictions, trainable_params, toolkit, expected_trainable_lora, adapters_identity_at_init, seconds)
G.check_submission("submission")

In [ ]:
print("Done")